In [2]:
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import haversine_distances

## METR-LA

In [3]:
df = pd.read_csv("graph_sensor_locations_lane.csv")
coords = df[['latitude', 'longitude']].values
lane_ids = df['lane_id'].astype(str).values
sensor_ids = df['sensor_id'].astype(str).values
sensor_ids_list = list(sensor_ids)
sensor_id_to_ind = {sid: i for i, sid in enumerate(sensor_ids_list)}

In [4]:
coords_rad = np.radians(coords)
dist_matrix = haversine_distances(coords_rad) * 6371000

In [5]:
dist_matrix

array([[    0.        ,  8555.48592357,  8528.37988279, ...,
        14910.57092153,  5193.66162423,  5821.54371064],
       [ 8555.48592357,     0.        ,    28.87147594, ...,
        22921.49108737, 13162.46580566,  7471.80061838],
       [ 8528.37988279,    28.87147594,     0.        , ...,
        22898.28887541, 13138.28398901,  7458.30085542],
       ...,
       [14910.57092153, 22921.49108737, 22898.28887541, ...,
            0.        ,  9813.86857953, 16496.20916531],
       [ 5193.66162423, 13162.46580566, 13138.28398901, ...,
         9813.86857953,     0.        ,  7546.57811257],
       [ 5821.54371064,  7471.80061838,  7458.30085542, ...,
        16496.20916531,  7546.57811257,     0.        ]])

In [6]:
lane_ids

array(['VF_R', 'GF_L', 'GF_R', 'HF_L', 'HF_R', 'HF_L', 'HF_R', 'GF_L',
       'SDF_R', 'SDF_R', 'HF_L', 'GF_R', 'HF_R', 'VF_L', 'SDF_R', 'HF_R',
       'ASF_L', 'HF_L', 'FF_L', 'HF_L', 'HF_R', 'FF_R', 'HF_R', 'GSF_R',
       'VF_R', 'GSF_R', 'SDF_L', 'GF_L', 'GF_R', 'ASF_R', 'ASF_L', 'HF_L',
       'HF_R', 'HF_R', 'SDF_L', 'FF_L', 'VF_R', 'VF_R', 'ASF_R', 'HF_L',
       'HF_R', 'VF_R', 'VF_R', 'VF_R', 'VF_L', 'VF_R', 'GF_L', 'VF_R',
       'ASF_R', 'GSF_L', 'FF_R', 'GSF_L', 'VF_R', 'VF_L', 'VF_R', 'GF_L',
       'GSF_R', 'HF_L', 'VF_L', 'SDF_R', 'VF_R', 'VF_L', 'VF_L', 'VF_L',
       'ASF_R', 'ASF_L', 'GF_R', 'VF_R', 'HF_R', 'VF_L', 'HF_R', 'HF_L',
       'VF_L', 'VF_R', 'HF_L', 'VF_L', 'VF_R', 'SDF_R', 'GF_R', 'GF_R',
       'HF_L', 'VF_L', 'VF_R', 'HF_R', 'SDF_R', 'GF_R', 'VF_L', 'VF_R',
       'SDF_R', 'SDF_L', 'HF_R', 'HF_L', 'GF_L', 'HF_R', 'HF_L', 'GSF_R',
       'HF_R', 'HF_L', 'HF_R', 'HF_L', 'VF_L', 'GSF_L', 'HF_R', 'HF_R',
       'SDF_L', 'GF_R', 'GF_R', 'GF_L', 'GF_R', 'GSF_

In [7]:
#KNN Adjacency

def knn_adjacency(dist_matrix, k=3):
    N = dist_matrix.shape[0]
    adj = np.zeros((N, N))
    for i in range(N):
        nearest = np.argsort(dist_matrix[i])[1:k+1] 
        adj[i, nearest] = 1
    return adj

In [8]:
#Radius Adjacency

def radius_adjacency(dist_matrix, radius_meters=1000):
    adj = (dist_matrix < radius_meters).astype(float)
    np.fill_diagonal(adj, 0)
    return adj

In [9]:
adj_knn = knn_adjacency(dist_matrix, k=3)
adj_radius = radius_adjacency(dist_matrix, radius_meters=1000)

In [10]:
adj_knn

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [11]:
adj_radius

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [12]:
#Convert to Pickle

def save_pickle(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

In [13]:
save_pickle("adj_mx_knn.pkl", (sensor_ids_list, sensor_id_to_ind, adj_knn))
save_pickle("adj_mx_radius.pkl", (sensor_ids_list, sensor_id_to_ind, adj_radius))

### Same Lane

In [14]:
dist_matrix

array([[    0.        ,  8555.48592357,  8528.37988279, ...,
        14910.57092153,  5193.66162423,  5821.54371064],
       [ 8555.48592357,     0.        ,    28.87147594, ...,
        22921.49108737, 13162.46580566,  7471.80061838],
       [ 8528.37988279,    28.87147594,     0.        , ...,
        22898.28887541, 13138.28398901,  7458.30085542],
       ...,
       [14910.57092153, 22921.49108737, 22898.28887541, ...,
            0.        ,  9813.86857953, 16496.20916531],
       [ 5193.66162423, 13162.46580566, 13138.28398901, ...,
         9813.86857953,     0.        ,  7546.57811257],
       [ 5821.54371064,  7471.80061838,  7458.30085542, ...,
        16496.20916531,  7546.57811257,     0.        ]])

In [15]:
same_lane_mask = lane_ids[:, None] == lane_ids[None, :]
filtered_dist_matrix = np.where(same_lane_mask, dist_matrix, np.inf)

In [16]:
print(same_lane_mask)

[[ True False False ... False False False]
 [False  True False ... False False False]
 [False False  True ... False False False]
 ...
 [False False False ...  True False False]
 [False False False ... False  True False]
 [False False False ... False False  True]]


In [17]:
filtered_dist_matrix

array([[ 0., inf, inf, ..., inf, inf, inf],
       [inf,  0., inf, ..., inf, inf, inf],
       [inf, inf,  0., ..., inf, inf, inf],
       ...,
       [inf, inf, inf, ...,  0., inf, inf],
       [inf, inf, inf, ..., inf,  0., inf],
       [inf, inf, inf, ..., inf, inf,  0.]])

In [18]:
def lane_knn_adjacency(filtered_dist, k=3):
    N = filtered_dist.shape[0]
    adj = np.zeros((N, N))
    for i in range(N):
        neighbors = np.argsort(filtered_dist[i])[:k+1]
        for j in neighbors:
            if i != j and filtered_dist[i, j] < np.inf:
                adj[i, j] = 1
    return adj


def lane_radius_adjacency(filtered_dist, radius=1000):
    adj = (filtered_dist < radius).astype(float)
    np.fill_diagonal(adj, 0)
    return adj

In [19]:
adj_knn_lane = lane_knn_adjacency(filtered_dist_matrix, k=3)
adj_radius_lane = lane_radius_adjacency(filtered_dist_matrix, radius=1000)

In [20]:
adj_knn_lane

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [21]:
adj_radius_lane

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [22]:
save_pickle("adj_mx_knn_lane.pkl", (sensor_ids_list, sensor_id_to_ind, adj_knn_lane))
save_pickle("adj_mx_radius_lane.pkl", (sensor_ids_list, sensor_id_to_ind, adj_radius_lane))

## PEMS-BAY

In [23]:
lane_df = pd.read_csv("pems_bay_sensor_locations_lane.csv")
sensor_ids = lane_df['sensor_id'].values
coords = lane_df[['latitude', 'longitude']].values
lane_ids = lane_df['lane_id'].values
sensor_ids_list = list(sensor_ids)
id_to_index = {int(sid): idx for idx, sid in enumerate(sensor_ids_list)}

In [24]:
coords_rad = np.radians(coords)
dist_matrix = haversine_distances(coords_rad) * 6371000

In [25]:
dist_matrix

array([[    0.        ,   553.34356055,   809.78926885, ...,
        18634.61582164, 18824.31102534, 18907.32783189],
       [  553.34356055,     0.        ,   257.08919759, ...,
        18136.78546191, 18309.22811638, 18390.03441636],
       [  809.78926885,   257.08919759,     0.        , ...,
        17917.30470262, 18079.82331615, 18159.32678356],
       ...,
       [18634.61582164, 18136.78546191, 17917.30470262, ...,
            0.        ,  1446.80929661,  1664.65367279],
       [18824.31102534, 18309.22811638, 18079.82331615, ...,
         1446.80929661,     0.        ,   222.00213727],
       [18907.32783189, 18390.03441636, 18159.32678356, ...,
         1664.65367279,   222.00213727,     0.        ]])

In [26]:
adj_knn_pems = knn_adjacency(dist_matrix, k=3)
adj_radius_pems = radius_adjacency(dist_matrix, radius_meters=1000)

In [27]:
adj_knn_pems

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.]])

In [28]:
adj_radius_pems

array([[0., 1., 1., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [1., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.]])

In [29]:
save_pickle("adj_mx_knn_pems.pkl", (sensor_ids_list, id_to_index, adj_knn_pems))
save_pickle("adj_mx_radius_pems.pkl", (sensor_ids_list, id_to_index, adj_radius_pems))

### Same Lane

In [30]:
same_lane_mask = lane_ids[:, None] == lane_ids[None, :]
filtered_dist_matrix = np.where(same_lane_mask, dist_matrix, np.inf)

In [31]:
print(same_lane_mask)

[[ True  True  True ... False False False]
 [ True  True  True ... False False False]
 [ True  True  True ... False False False]
 ...
 [False False False ...  True  True  True]
 [False False False ...  True  True  True]
 [False False False ...  True  True  True]]


In [32]:
filtered_dist_matrix

array([[   0.        ,  553.34356055,  809.78926885, ...,           inf,
                  inf,           inf],
       [ 553.34356055,    0.        ,  257.08919759, ...,           inf,
                  inf,           inf],
       [ 809.78926885,  257.08919759,    0.        , ...,           inf,
                  inf,           inf],
       ...,
       [          inf,           inf,           inf, ...,    0.        ,
        1446.80929661, 1664.65367279],
       [          inf,           inf,           inf, ..., 1446.80929661,
           0.        ,  222.00213727],
       [          inf,           inf,           inf, ..., 1664.65367279,
         222.00213727,    0.        ]])

In [33]:
adj_knn_lane_pems = lane_knn_adjacency(filtered_dist_matrix, k=3)
adj_radius_lane_pems = lane_radius_adjacency(filtered_dist_matrix, radius=1000)

In [34]:
adj_knn_lane_pems

array([[0., 1., 1., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [1., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 1., 0., 1.],
       [0., 0., 0., ..., 1., 1., 0.]])

In [35]:
adj_radius_lane_pems

array([[0., 1., 1., ..., 0., 0., 0.],
       [1., 0., 1., ..., 0., 0., 0.],
       [1., 1., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.],
       [0., 0., 0., ..., 0., 1., 0.]])

In [36]:
save_pickle("adj_mx_knn_lane_pems.pkl", (sensor_ids_list, id_to_index, adj_knn_lane_pems))
save_pickle("adj_mx_radius_lane_pems.pkl", (sensor_ids_list, id_to_index, adj_radius_lane_pems))